In [ ]:
import sys
import os
import gc
from pathlib import Path  # noqa: F401

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import matplotlib.pyplot as plt  # noqa: E402
import seaborn as sns  # noqa: E402

from src.preprocessing.pipeline import DatasetHandler  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ExperimentNames,
    CoordinateSystems,
    PreprocessedDataVariants,
    ExcludedICsMetadata,
    SingleDataMetadata,
    MusicTypeVariants,
    ConditionVariants,
)
from src.visualization.preprocessing_plots import DatasetPlotter  # noqa: E402
from src.preprocessing.ica import IC_LABEL_CLASSES_ORDER  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
print("Setup complete.")

# EEG Data Inspection — Preprocessing Stages

This notebook provides an interactive tool to inspect raw EEG data at **different stages**
of the preprocessing pipeline (before ICA, after ICA, cropped).

Works for **either experiment** (`EXPERIMENT` in the *Configuration* cell):
* `PSILO_MUSIC` — music listening (CLASSIC / PSYTRANCE)
* `ASSR` — auditory steady-state response (no music dimension; the music type is
  fixed to the `MusicTypeVariants.ASSR` placeholder)

Capabilities:
* Browse dataset metadata and per-recording statistics
* Inspect raw time series for a selected recording
* Plot topomaps (per frequency band and broadband)
* Plot spectral power distributions
* Inspect excluded ICA components and their probabilities
* Visualise sensor arrangement

> **Parameters to tweak:** `EXPERIMENT`, `PARTICIPANT_ID`, `MUSIC_TYPE`, `CONDITION`,
> `DATA_STAGE` in the *Configuration* cell below.

## Configuration

In [ ]:
# ── Experiment selection ──────────────────────────────────────────────────────
# PSILO_MUSIC: music-listening experiment (CLASSIC / PSYTRANCE).
# ASSR:        auditory steady-state response experiment. It has no music
#              dimension, so MUSIC_TYPE below is overridden with the placeholder
#              MusicTypeVariants.ASSR.
# Everything downstream (metadata, preprocessing stages, ICA products) is loaded
# from the selected experiment's data directories.
EXPERIMENT = ExperimentNames.ASSR

# ── Recording selection ───────────────────────────────────────────────────────
PARTICIPANT_ID = "035"  # participant numeric ID from metadata
MUSIC_TYPE = MusicTypeVariants.CLASSICAL  # ignored for the ASSR experiment
CONDITION = ConditionVariants.PLACEBO

if EXPERIMENT == ExperimentNames.ASSR:
    # ASSR has no music dimension; uses a single placeholder "music type".
    MUSIC_TYPE = MusicTypeVariants.ASSR

# ── Data stage to inspect ─────────────────────────────────────────────────────
# Choose one of: RAW_BEFORE_ICA, RAW_AFTER_ICA, RAW_CROPPED
DATA_STAGE = PreprocessedDataVariants.RAW_AFTER_ICA

# ── Spectral plot max frequency ───────────────────────────────────────────────
FMAX = 125  # Hz

# ── Plot saving ──────────────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "00-preprocessing"
    / "plots"
    / "eeg_data_inspection"
    / EXPERIMENT.value
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Experiment: {EXPERIMENT.value}")
print(f"Recording : {PARTICIPANT_ID} / {MUSIC_TYPE.value} / {CONDITION.value}")
print(f"Plots will be saved to: {PLOTS_DIR}")

## Dataset Initialisation

In [ ]:
dataset_handler = DatasetHandler(
    EXPERIMENT, CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS
)
dataset_metadata = dataset_handler.dataset_metadata
excluded_ics_metadata = dataset_handler.dataset_excluded_ics_metadata
print(f"Experiment {EXPERIMENT.value}: {len(dataset_metadata)} recordings")

## Metadata Overview

Browse all recordings and their metadata fields.

In [ ]:
# Display full metadata table
print(f"Total recordings: {len(dataset_metadata)}")
print(f"Columns: {list(dataset_metadata.columns)}")
dataset_metadata.head(20)

## Recording Selection

Select the recording for a given participant / music type / condition combination.

In [ ]:
recording_df = dataset_metadata[
    (dataset_metadata[SingleDataMetadata.PARTICIPANT_ID] == PARTICIPANT_ID)
    & (dataset_metadata[SingleDataMetadata.MUSIC_TYPE] == MUSIC_TYPE)
    & (dataset_metadata[SingleDataMetadata.CONDITION] == CONDITION)
]

if recording_df.empty:
    # Participant coverage differs between experiments — list what is available
    # for the selected group to make the mismatch obvious.
    available = sorted(
        dataset_metadata[
            (dataset_metadata[SingleDataMetadata.MUSIC_TYPE] == MUSIC_TYPE)
            & (dataset_metadata[SingleDataMetadata.CONDITION] == CONDITION)
        ][SingleDataMetadata.PARTICIPANT_ID].unique()
    )
    raise ValueError(
        f"No {EXPERIMENT.value} recording found for participant {PARTICIPANT_ID}, "
        f"{MUSIC_TYPE.value}, {CONDITION.value}. "
        f"Available participants for this group: {available}"
    )

original_filename = recording_df[SingleDataMetadata.FILENAME].iloc[0]
print(f"Selected recording: {original_filename}")
recording_df

## Load EEG Data

Load the raw EEG data for the selected recording and stage (`DATA_STAGE`).

In [ ]:
# Free memory from any previously loaded data
try:
    del raw_data
    gc.collect()
except NameError:
    pass

raw_data = dataset_handler.load_data_file(
    original_filename,
    is_processed=True,
    processed_data_type=DATA_STAGE,
)
print(f"Loaded {DATA_STAGE.value}: {raw_data}")
print(
    f"Duration: {raw_data.times[-1]:.1f} s  |  Sampling rate: {raw_data.info['sfreq']} Hz"
)
print(f"Channels: {len(raw_data.ch_names)}")

In [ ]:
ann = raw_data.annotations
for onset, dur, desc in zip(ann.onset, ann.duration, ann.description):
    print(f"{onset:9.3f}  {dur:6.3f}  {desc}")

## Raw Time Series

Interactive MNE viewer showing the EEG time series.

In [ ]:
raw_data.plot(n_channels=50, block=False)

## Spectral Power

Average power spectral density across all EEG channels.

In [ ]:
DatasetPlotter.plot_raw_dataseries(
    raw_data,
    save_fig=original_filename.split(".")[0] if SAVE_PLOTS else "",
    plot_variant="power_spectrum",
    variant_name=DATA_STAGE,
    custom_full_path=str(
        PLOTS_DIR / f"power_spectrum_{original_filename.split('.')[0]}.png"
    )
    if SAVE_PLOTS
    else "",
    fmax=FMAX,
    title=f"Power Spectrum — {original_filename} ({DATA_STAGE.value})",
)

## Topomaps

Spatial distribution of RMS power per frequency band and broadband.

In [ ]:
DatasetPlotter.plot_raw_dataseries(
    raw_data,
    save_fig=original_filename.split(".")[0] if SAVE_PLOTS else "",
    plot_variant="topomap",
    variant_name=DATA_STAGE,
    custom_full_path=str(PLOTS_DIR / f"topomap_{original_filename.split('.')[0]}.png")
    if SAVE_PLOTS
    else "",
    title=f"Topomap — {original_filename} ({DATA_STAGE.value})",
)

## Compare Before / After ICA

Load both pre-ICA and post-ICA versions of the same recording to compare.

In [ ]:
try:
    del before_ica, after_ica
    gc.collect()
except NameError:
    pass

before_ica = dataset_handler.load_data_file(
    original_filename,
    is_processed=True,
    processed_data_type=PreprocessedDataVariants.RAW_BEFORE_ICA,
)
after_ica = dataset_handler.load_data_file(
    original_filename,
    is_processed=True,
    processed_data_type=PreprocessedDataVariants.RAW_AFTER_ICA,
)
print("Before ICA:", before_ica)
print("After  ICA:", after_ica)

In [ ]:
# Plot power spectra side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
before_ica.compute_psd(fmax=FMAX).plot(
    average=True,
    picks="data",
    exclude="bads",
    amplitude=False,
    axes=axes[0],
    show=False,
)
axes[0].set_title("Before ICA")
after_ica.compute_psd(fmax=FMAX).plot(
    average=True,
    picks="data",
    exclude="bads",
    amplitude=False,
    axes=axes[1],
    show=False,
)
axes[1].set_title("After ICA")
plt.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / f"compare_ica_{original_filename.split('.')[0]}.png", dpi=150
    )
plt.show()

## Excluded ICA Components

Inspect the ICA components that were removed during preprocessing for this recording.

In [ ]:
keys_of_interest = [
    ExcludedICsMetadata.IC_ID.value,
    ExcludedICsMetadata.IC_CATEGORY.value,
    ExcludedICsMetadata.MAIN_PROBABILITY.value,
    ExcludedICsMetadata.TOTAL_ICS.value,
]
excluded_df = excluded_ics_metadata[
    excluded_ics_metadata[ExcludedICsMetadata.ORIGINAL_FILENAME.value]
    == original_filename
][keys_of_interest]

print(f"Excluded ICs for {original_filename}: {len(excluded_df)}")
excluded_df

## IC Probability Distribution

For a specific excluded IC, show its label probabilities across all ICA classes.

In [ ]:
# Set EXCLUDED_IC_IDX to an IC id shown in the table above
EXCLUDED_IC_IDX = 1

ic_row = excluded_df[excluded_df[ExcludedICsMetadata.IC_ID.value] == EXCLUDED_IC_IDX]
ic_probabilities = dataset_handler.load_data_file(
    original_filename,
    is_processed=True,
    processed_data_type=PreprocessedDataVariants.IC_PROBABILITIES,
)

print(f"Label: {ic_row[ExcludedICsMetadata.IC_CATEGORY.value].values}")
for i, label in enumerate(IC_LABEL_CLASSES_ORDER):
    print(f"  {label}: {ic_probabilities[EXCLUDED_IC_IDX, i]:.4f}")

## IC Topography

Plot the scalp topography of a selected excluded IC component.

In [ ]:
try:
    del ica_components
    gc.collect()
except NameError:
    pass

ica_components = dataset_handler.load_data_file(
    original_filename,
    is_processed=True,
    processed_data_type=PreprocessedDataVariants.ICA_COMPONENTS,
)

DatasetPlotter.plot_raw_dataseries(
    ica_components,
    plot_variant="topomap",
    is_excluded=True,
    excluded_ic_id=EXCLUDED_IC_IDX,
    variant_name=PreprocessedDataVariants.RAW_EXCLUDED_IC,
    custom_full_path=str(
        PLOTS_DIR / f"ic_topo_{original_filename.split('.')[0]}_IC{EXCLUDED_IC_IDX}.png"
    )
    if SAVE_PLOTS
    else "",
    title=f"IC {EXCLUDED_IC_IDX} topography",
)

## Sensor Distribution

Visualise the electrode positions for the loaded recording.

In [ ]:
raw_data.plot_sensors(show_names=True)